# Analyse des clients — Online Retail II

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.load_data import charger_ventes_en_ligne
from src.clean_transactions import nettoyer_transactions
from src.build_orders import construire_commandes
from src.calculate_kpis import ajouter_variables_temporelles
from src.clean_customers import construire_clients, calculer_delai_moyen_reachat

donnees = charger_ventes_en_ligne()
donnees = nettoyer_transactions(donnees)
donnees = ajouter_variables_temporelles(donnees)
commandes = construire_commandes(donnees)
clients = construire_clients(donnees, commandes)

clients.shape

(5942, 14)

## Combien de clients uniques ?

In [2]:
n_clients = clients.shape[0]
print("Clients uniques :", n_clients)

Clients uniques : 5942


## Achat unique vs commandes multiples

In [3]:
clients_achat_unique = (clients["number_of_orders"] == 1).sum()
clients_multi_commandes = (clients["number_of_orders"] > 1).sum()

print(f"Achat unique : {clients_achat_unique} ({clients_achat_unique / n_clients:.1%})")
print(f"Plusieurs commandes : {clients_multi_commandes} ({clients_multi_commandes / n_clients:.1%})")

Achat unique : 1461 (24.6%)
Plusieurs commandes : 4481 (75.4%)


## Délai moyen entre deux achats

In [4]:
delai_moyen = calculer_delai_moyen_reachat(commandes)
print(f"Délai moyen de réachat : {delai_moyen:.1f} jours")

Délai moyen de réachat : 41.7 jours


## Top 10 clients par revenu net

In [5]:
clients.sort_values("net_revenue", ascending=False).head(10)[
    ["customer_id", "country", "net_revenue", "number_of_orders"]
]

,customer_id,country,net_revenue,number_of_orders
5756,18102.0,United Kingdom,598215.22,153
2300,14646.0,Netherlands,523342.07,164
1810,14156.0,Ireland,296564.69,202
2565,14911.0,Ireland,270248.53,510
5104,17450.0,United Kingdom,233579.39,61
1348,13694.0,United Kingdom,190825.52,164
5165,17511.0,United Kingdom,171885.98,85
69,12415.0,Australia,143269.29,33
4338,16684.0,United Kingdom,141502.25,65
2715,15061.0,United Kingdom,136391.48,138


## Top 10 clients par fréquence de commande

In [6]:
clients.sort_values("number_of_orders", ascending=False).head(10)[
    ["customer_id", "country", "number_of_orders", "net_revenue"]
]

,customer_id,country,number_of_orders,net_revenue
2565,14911.0,Ireland,510,270248.53
402,12748.0,United Kingdom,365,49970.13
5495,17841.0,United Kingdom,289,69516.19
2965,15311.0,United Kingdom,270,113513.07
2260,14606.0,United Kingdom,259,30094.38
743,13089.0,United Kingdom,247,113214.19
1810,14156.0,Ireland,202,296564.69
2181,14527.0,United Kingdom,190,25774.54
1348,13694.0,United Kingdom,164,190825.52
2300,14646.0,Netherlands,164,523342.07


## Clients ayant récemment cessé d'acheter

Seuil arbitraire de départ : aucun achat depuis plus de 90 jours (avant la
dernière date du dataset). Ce seuil sera affiné lors de la segmentation RFM
(Phase 6).

In [7]:
seuil_inactivite = 90
clients_inactifs = clients[clients["recency_days"] > seuil_inactivite]

print(
    f"Clients inactifs depuis plus de {seuil_inactivite} jours : "
    f"{clients_inactifs.shape[0]} sur {n_clients} "
    f"({clients_inactifs.shape[0] / n_clients:.1%})"
)

clients_inactifs.sort_values("net_revenue", ascending=False).head(10)[
    ["customer_id", "country", "net_revenue", "recency_days"]
]

Clients inactifs depuis plus de 90 jours : 3024 sur 5942 (50.9%)


,customer_id,country,net_revenue,recency_days
4408,16754.0,United Kingdom,56560.58,372
5504,17850.0,United Kingdom,55703.13,302
747,13093.0,United Kingdom,54073.73,267
1556,13902.0,Denmark,30411.26,632
1456,13802.0,United Kingdom,25491.56,138
136,12482.0,Sweden,21893.53,450
3403,15749.0,United Kingdom,21535.90,235
3462,15808.0,United Kingdom,18248.83,306
681,13027.0,United Kingdom,17148.00,114
4207,16553.0,United Kingdom,16555.87,163


## Part du chiffre d'affaires générée par les 10% meilleurs clients

In [8]:
n_top10 = max(int(n_clients * 0.10), 1)
top10_revenue = (
    clients.sort_values("net_revenue", ascending=False)
    .head(n_top10)["net_revenue"]
    .sum()
)
part_top10 = top10_revenue / clients["net_revenue"].sum()

print(
    f"Les 10% meilleurs clients ({n_top10} sur {n_clients}) "
    f"génèrent {part_top10:.1%} du chiffre d'affaires net"
)

Les 10% meilleurs clients (594 sur 5942) génèrent 63.7% du chiffre d'affaires net


## Les gros clients sont-ils principalement des grossistes ?

Comparaison de la quantité moyenne par commande entre les 10% meilleurs
clients (par revenu) et l'ensemble des clients. Une quantité par commande
nettement plus élevée chez les meilleurs clients suggère une clientèle de
grossistes plutôt que de particuliers.

In [ ]:
top10_clients = clients.sort_values("net_revenue", ascending=False).head(n_top10)

print(
    "Quantité moyenne/commande — top 10% clients :",
    round(top10_clients["quantite_moyenne_par_commande"].mean(), 1),
)
print(
    "Quantité moyenne/commande — ensemble des clients :",
    round(clients["quantite_moyenne_par_commande"].mean(), 1),
)

## Export

In [ ]:
clients.to_csv(Path("../data/processed/customers.csv"), index=False)
print("Fichier exporté : data/processed/customers.csv")